In [1]:
import torch # pytorch library
import torch.nn as nn # pyt's nn module
import torch.optim as optim # import the optimization algos like adam
from torchvision import datasets, transforms # ready-made dsets like mnist, transforms includes tools that preprocess the data (normalize and convert to tensor)
from torch.utils.data import DataLoader # presents the data in batches

In [13]:
# mnist

transform = transforms.Compose([ # combines preprocessing steps into a pipeline
    transforms.ToTensor(), # converts image to tensor, changes pixel values [0.0, 1.0]
    transforms.Normalize((0.1307,), (0.3081,)) # "normalizes" the mnist data, makes training faster
])

training_data = datasets.MNIST('./data', train=True, download=True, transform=transform) # load mnist dataset, "train=True" is training split, "download=True" is downloads if not present, "transform=transform" does the preprocessing
testing_data = datasets.MNIST('./data', train=False, transform=transform) # loads the test dataset 10k images, no download needed if it is already present

training_loader = DataLoader(training_data, batch_size=64, shuffle=True) # wraps training data to batches of 64, "shuffle=True" randomizes data for each epoch
testing_loader  = DataLoader(testing_data,  batch_size=1000) # loads test data to batches of 1k, no shuffle (order doesnt matter)

In [10]:
# the model

class MNISTNet(nn.Module): # the nn class
    def __init__(self): # defines the layers
        super().__init__() # calling the parent class constructor
        self.fc1 = nn.Linear(28*28, 128) # fully connected layer - 784 inputs, output of 128 neurons
        self.fc2 = nn.Linear(128, 64) # input 128 -> output 64
        self.fc3 = nn.Linear(64, 10) # 64 -> 10, 10 outputs 10 digit classes
        self.relu = nn.ReLU() # applies the relu activation function

    # the forward pass
    def forward(self, x):
        x = x.view(-1, 28*28)   # flatten image from (batch, 1, 28, 28) -> (batch, 784)
        x = self.relu(self.fc1(x)) # first linear layer and apply relu activation
        x = self.relu(self.fc2(x)) # second layer and relu
        return self.fc3(x) # no relu and outputs the raw scores

model = MNISTNet() # isntance of the network
optimizer = optim.Adam(model.parameters(), lr=1e-3) # adam optimizer updates weights (learning rate of 0.001)
criterion = nn.CrossEntropyLoss() # loss function combines softmax and negative log-likehood loss

In [12]:
# the training part

for epoch in range(5): # train for 5 passes
  model.train() # training mode
  for images, labels in training_loader: # loops through the batches
    optimizer.zero_grad() # clear prev gradients to prevent accumulation
    output = model(images) # forward pass
    loss = criterion(output, labels) # compute error between predictions and actual labels
    loss.backward() # backprop
    optimizer.step() # update the weights

  # eval

  model.eval() # eval mode
  correct = 0 # counter of correct predictions
  with torch.no_grad(): # disable gradient tracking, speeds up evals
    for images, labels in testing_loader: #loops over test data
      preds = model(images).argmax(dim=1) # get predicted class
      correct += (preds == labels).sum().item() # counts correct predictions
  print(f"Epoch {epoch+1}: Test Accuracy = {correct / len(testing_data):.2%}")

Epoch 1: Test Accuracy = 96.93%
Epoch 2: Test Accuracy = 97.22%
Epoch 3: Test Accuracy = 97.26%
Epoch 4: Test Accuracy = 97.54%
Epoch 5: Test Accuracy = 97.60%
